<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/GTT_LEFM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
TEST WITH 6 PRIMES (2,3,5,7,11,13)
"""

import math
import mpmath
import numpy as np

mpmath.mp.dps = 50

def generate_primes(limit=20000):
    sieve = [True] * (limit + 1)
    sieve[0] = sieve[1] = False
    for p in range(2, int(limit**0.5) + 1):
        if sieve[p]:
            for i in range(p * p, limit + 1, p):
                sieve[i] = False
    return [p for p in range(2, limit + 1) if sieve[p]]

ALL_PRIMES = generate_primes(20000)

GREEN_TAO = {
    3: [3, 5, 7],
    4: [5, 11, 17, 23],
    5: [5, 17, 29, 41, 53],
    6: [7, 37, 67, 97, 127, 157],
}

def lefm_symbol(sigma, gamma, primes):
    s = mpmath.mpc(sigma, gamma)
    prod = mpmath.mpc(1, 0)
    for p in primes:
        prod *= 1.0 / (1.0 - mpmath.power(p, -s))
    return prod

print("=" * 100)
print("TESTING WITH 6 PRIMES (2,3,5,7,11,13)")
print("=" * 100)

primes_used = ALL_PRIMES[:6]  # First 6 primes
print(f"Primes used: {primes_used}")
print()

ref_val = float(abs(lefm_symbol(0.5, math.log(2), primes_used)))

# C values around the optimal range from 5-prime test
C_values = [4.0, 4.2, 4.4, 4.6, 4.8, 5.0, 5.2, 5.4, 5.5, 5.6, 5.8, 6.0]

print(f"{'C':<8} {'k=3':<12} {'k=4':<12} {'k=5':<12} {'k=6':<12}")
print("-" * 60)

for C in C_values:
    results = {}
    for k, prog in GREEN_TAO.items():
        responses = []
        for p in prog:
            gamma = math.sqrt(max(math.log(p), 1e-10)) / C
            mag = float(abs(lefm_symbol(0.5, gamma, primes_used)))
            mag = mag / ref_val
            responses.append(mag)
        avg = np.mean(responses)
        coh = 1.0 - 1.0 / (1.0 + avg)
        results[k] = coh

    print(f"{C:<8.2f} {results[3]:<12.6f} {results[4]:<12.6f} {results[5]:<12.6f} {results[6]:<12.6f}")

print("\n" + "=" * 100)
print("FINDING BEST C FOR 6 PRIMES")
print("=" * 100)

# Find C that minimizes max error to target
TARGET = {3: 0.8731, 4: 0.8120, 5: 0.8012, 6: 0.7442}

best_c = None
best_error = float('inf')
best_results = None

for C in C_values:
    results = {}
    for k, prog in GREEN_TAO.items():
        responses = []
        for p in prog:
            gamma = math.sqrt(max(math.log(p), 1e-10)) / C
            mag = float(abs(lefm_symbol(0.5, gamma, primes_used)))
            mag = mag / ref_val
            responses.append(mag)
        avg = np.mean(responses)
        coh = 1.0 - 1.0 / (1.0 + avg)
        results[k] = coh

    errors = [abs(results[k] - TARGET[k]) for k in [3,4,5,6]]
    max_error = max(errors)

    if max_error < best_error:
        best_error = max_error
        best_c = C
        best_results = results

print(f"\nBest C = {best_c:.2f}")
print(f"Results: 3:{best_results[3]:.6f}, 4:{best_results[4]:.6f}, 5:{best_results[5]:.6f}, 6:{best_results[6]:.6f}")
print(f"Max error: {best_error:.6f}")

print("\n" + "=" * 100)
print("COMPARISON: 5 primes vs 6 primes")
print("=" * 100)

print("\n5 primes (C=4.40):")
print(f"  3:0.855051, 4:0.813226, 5:0.788241, 6:0.746045")

print(f"\n6 primes (C={best_c:.2f}):")
print(f"  3:{best_results[3]:.6f}, 4:{best_results[4]:.6f}, 5:{best_results[5]:.6f}, 6:{best_results[6]:.6f}")

print(f"\nPaper targets:")
print(f"  3:0.8731, 4:0.8120, 5:0.8012, 6:0.7442")

TESTING WITH 6 PRIMES (2,3,5,7,11,13)
Primes used: [2, 3, 5, 7, 11, 13]

C        k=3          k=4          k=5          k=6         
------------------------------------------------------------
4.00     0.874553     0.823601     0.793270     0.739760    
4.20     0.881924     0.836627     0.809273     0.761016    
4.40     0.888182     0.847706     0.822973     0.779392    
4.60     0.893537     0.857189     0.834761     0.795323    
4.80     0.898155     0.865358     0.844959     0.809183    
5.00     0.902163     0.872438     0.853827     0.821285    
5.20     0.905664     0.878609     0.861577     0.831894    
5.40     0.908740     0.884017     0.868383     0.841231    
5.50     0.910140     0.886474     0.871478     0.845481    
5.60     0.911457     0.888782     0.874389     0.849480    
5.80     0.913869     0.893000     0.879712     0.856797    
6.00     0.916020     0.896751     0.884450     0.863312    

FINDING BEST C FOR 6 PRIMES

Best C = 4.00
Results: 3:0.874553, 4:0.8236